# Packages import

In [ ]:
import yaml
import os
import re
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'pandas'

# Apollo Scraper

In [3]:
with open("conifg.yaml","r", encoding = "UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config["credentials"]['user']
password = config["credentials"]['password']
credentials = HTTPBasicAuth(username, password)

NameError: name 'yaml' is not defined

In [14]:
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

200


In [15]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [16]:
groupe = page_dom.select_one("div.grupa").get_text(strip=True)
print(groupe)

ZICSS1-1211


In [1]:
classes_tag = page_dom.select_one("table")
with open("temp.html","w",encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html",encoding="UTF-8", header = 0)[0]
os.remove("temp.html")

NameError: name 'page_dom' is not defined

In [ ]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

In [ ]:
classes[['Day','Start time','End time','Duration']] = classes['Dzień, godzina'].str.split(' ',expand = True)

In [ ]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [ ]:
classes = classes.drop(['Dzień, godzina','hyphen'], axis = 1)

In [ ]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.)*",
    r"\1",
    regex=True
)

In [18]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [19]:
classes.to_csv(f"schedules/{groupe}.csv")